# Intelligent Plant Disease Detection System (IPDDS)
## Targeted Model Training: Potato & Tomato Leaf Diseases
This notebook trains a deep Convolutional Neural Network (CNN) specifically on **Potato** and **Tomato** crop diseases from the **PlantVillage** dataset.

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print(f"TensorFlow Version: {tf.__version__}")

### 1. Dataset Paths & Target Classes
The PlantVillage dataset contains 38 total crop categories. Here we filter exclusively for **13 Potato & Tomato** classes.

In [ ]:
# Target Tomato & Potato Categories
POTATO_TOMATO_CLASSES = [
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___healthy",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus"
]

# Locate dataset directory
cache_dir = os.path.expanduser(r"~/.cache/kagglehub/datasets/mohitsingh1804/plantvillage/versions/1/PlantVillage")
train_path = os.path.join(cache_dir, "train")
val_path = os.path.join(cache_dir, "val")

print(f"Train directory : {train_path} (Exists: {os.path.exists(train_path)})")
print(f"Val directory   : {val_path} (Exists: {os.path.exists(val_path)})")

### 2. Loading Filtered Image Datasets
By passing `class_names=POTATO_TOMATO_CLASSES`, TensorFlow automatically ignores other plant species and loads only the 13 Potato and Tomato classes.

In [ ]:
IMG_SIZE = (256, 256)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    labels="inferred",
    class_names=POTATO_TOMATO_CLASSES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_path,
    labels="inferred",
    class_names=POTATO_TOMATO_CLASSES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Optimize I/O pipeline
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

print(f"Classes loaded ({len(train_ds.class_names)} total): {train_ds.class_names}")

### 3. Model Architecture with Data Augmentation

In [ ]:
inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

# Data Augmentation
x = layers.RandomFlip("horizontal_and_vertical")(inputs)
x = layers.RandomRotation(0.15)(x)
x = layers.RandomZoom(0.1)(x)
x = layers.Rescaling(1./255)(x)

# Feature Extractor
x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(256, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

# Classification Head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(POTATO_TOMATO_CLASSES), activation="softmax")(x)

model = models.Model(inputs=inputs, outputs=outputs, name="IPDDS_CNN")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

### 4. Training & Saving Model Weights

In [ ]:
os.makedirs("../model", exist_ok=True)
checkpoint_cb = callbacks.ModelCheckpoint(
    "../model/disease_model.h5",
    save_best_only=True,
    monitor="val_accuracy",
    mode="max"
)
early_stop_cb = callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[checkpoint_cb, early_stop_cb]
)

# Save modern .keras format as well
model.save("../model/disease_model.keras")
print("Model successfully saved to ../model/disease_model.h5 and ../model/disease_model.keras")